> **교육 목표** 누수를 체험하고, 분석 단위·기준 시점·라벨을 정해 Data Mart를 만들어 같은 모델의 성능을 올립니다.
>
> **핵심 내용** 누수 컬럼의 가짜 고성능 → 파생변수 → 설비 과거 기록 피처 → Data Mart 체크리스트 → 같은 모델, 다른 데이터 비교 → 저장

In [1]:
%%html
<style>
table {
    float: left;
    clear: both;
}
</style><!-- Markdown 왼쪽 정렬 -->

# D3_03 Data Mart 만들기

> **오늘 질문:** 모델을 바꾸지 않고, 데이터만 바꿔서 성능을 올릴 수 있을까?

Data Mart는 '예측하기 위한 표'입니다. 세 가지를 정하고 시작합니다.

| 질문 | 오늘의 답 |
|---|---|
| 분석 단위 (1행은?) | 생산 기록 1건 |
| 기준 시점 (언제 예측?) | 생산 직전. 그 시점에 알 수 있는 값만 사용 |
| 라벨 (무엇이 고장?) | `failure` = 1 |

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비 📋

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

df = pd.read_parquet("../../data/factory_table.parquet")   # 시간 순 정렬
cut = int(len(df) * 0.8)                                    # 앞 80% 학습 / 마지막 20% 평가
base = ["air_temp", "process_temp", "rpm", "torque", "tool_wear"]

## 1. 누수 체험 📋

고장 유형 컬럼(`twf` ~ `rnf`)을 입력에 넣으면 어떻게 될까요? 이 컬럼들은 **고장이 난 뒤에** 기록되는 값입니다.

In [ ]:
leak_cols = base + ["twf", "hdf", "pwf", "osf", "rnf"]
model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(df[leak_cols].iloc[:cut], df["failure"].iloc[:cut])
pred = model.predict(df[leak_cols].iloc[cut:])
print(f"누수 컬럼 포함 F1: {f1_score(df['failure'].iloc[cut:], pred):.2f}")

**Q1.** F1이 거의 1입니다. 이 모델을 현장에 배포하면 어떻게 될까요?

> 답:


## 2. 파생변수 만들기 ✍️

현장 지식을 숫자로 바꿉니다. 모두 **생산 시점에 알 수 있는 값**으로만 계산합니다.

| 새 컬럼 | 계산 | 현장 의미 |
|---|---|---|
| `temp_diff` | 공정 온도 − 공기 온도 | 열이 잘 빠지는가 |
| `power` | 토크 × 회전수 | 설비에 걸리는 동력 |
| `strain` | 공구 마모 × 토크 | 닳은 공구에 걸리는 부하 |
| `grade_num` | L·M·H → 0·1·2 | 제품 등급 |

In [ ]:
# 힌트: df["새컬럼"] = 계산식, df["컬럼"].map({"L": 0, ...})

## 3. 설비의 과거 기록 피처 ✍️

"이 설비가 지금까지 몇 번 고장 났나"는 쓸 만한 정보입니다. 단, **지금 기록의 고장은 빼야** 합니다. 지금 고장은 예측할 대상이기 때문입니다.

> 새 컬럼: `equip_past_fail`

In [ ]:
# 힌트: df.groupby("설비")["failure"].cumsum() - df["failure"]

## 4. Data Mart 정리와 체크리스트 ✍️

모델에 넣을 컬럼만 고르고, 체크리스트를 확인합니다.

> 변수 이름: `mart_cols`, `mart`

## 5. 같은 모델, 다른 데이터 ✍️

모델은 1교시와 똑같은 깊이 5 나무입니다. 입력만 기본 센서 5개 → Data Mart로 바꿔 비교합니다.

> 변수 이름: `tree`

In [ ]:
# 힌트: 같은 모델을 mart_cols로 학습

1교시 기준(센서 5개, 같은 모델)과 비교합니다. 📋

In [ ]:
tree_base = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_base.fit(df[base].iloc[:cut], df["failure"].iloc[:cut])
pred_base = tree_base.predict(df[base].iloc[cut:])
print(f"센서 5개   정밀도 {precision_score(y_test, pred_base):.2f} / 재현율 {recall_score(y_test, pred_base):.2f} / F1 {f1_score(y_test, pred_base):.2f}")

# 어떤 피처가 많이 쓰였나
pd.Series(tree.feature_importances_, index=mart_cols).sort_values(ascending=False).round(3)

## 6. Data Mart 저장 📋

만드는 코드와 결과를 함께 남겨 두면 언제든 다시 만들 수 있습니다.

In [ ]:
mart.to_parquet("data_mart.parquet", index=False)

## 정리 💬

**Q2.** 같은 모델인데 F1이 얼마나 바뀌었나요? 가장 중요하게 쓰인 피처는 무엇이고, 왜 도움이 됐을까요?

> 답:
